# 第8章　医療画像の座標系と位置合わせ ― 経過比較とマルチモダリティ**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 8.3　解像度をそろえる ― リサンプリング

In [ ]:
spacing = img.header.get_zooms()[:3]        # 各軸のボクセル間隔（mm）assert all(s > 0 and np.isfinite(s) for s in spacing), f"不正なspacing: {spacing}"

## 数字で追う ― リサンプリングの中身は「近い隣ほど強い」線形補間

In [ ]:
import numpy as npQ = np.array([[10., 20.], [30., 40.]])def bilinear(Q, r, c):        # (r, c) は元の格子上の実数座標    r0, c0 = int(np.floor(r)), int(np.floor(c))    b, a = r - r0, c - c0     # 基準格子点からの行・列の相対位置(0〜1)    return (Q[r0, c0]*(1-a)*(1-b) + Q[r0, c0+1]*a*(1-b)          + Q[r0+1, c0]*(1-a)*b   + Q[r0+1, c0+1]*a*b)print(bilinear(Q, 0.3, 0.7))   # 23.0

In [ ]:
from scipy.ndimage import zoom# 0.7mm → 1.0mm へ。倍率 = 0.7/1.0 = 0.7（画素数は減る）img_rs   = zoom(image, zoom=0.7, order=1)   # 画像は線形補間label_rs = zoom(label, zoom=0.7, order=0)   # ラベルは最近傍(IDを壊さない)

In [ ]:
Spacingd(keys=["image", "label"], pixdim=(1, 1, 1),         mode=("bilinear", "nearest"))     # 画像は滑らかに、ラベルは最近傍で

## 具体例で見る ― リサンプリングを怠ると、何が起きるか

In [ ]:
# 「1ボクセル = 1mm³」に揃える。以後はボクセル数がそのまま体積(mm³)になる。from monai.transforms import Spacingdtf = Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0),              mode=("bilinear", "nearest"))   # 画像は線形、ラベルは最近傍（鉄則）

## 場面によって、やることがまるで違う

In [ ]:
import SimpleITK as sitkdef phase_shift_mm(fixed_path, moving_path):    """2つの相のあいだの平行移動量(mm)を測る。まず剛体で十分。"""    fixed  = sitk.ReadImage(fixed_path,  sitk.sitkFloat32)   # 例：門脈相    moving = sitk.ReadImage(moving_path, sitk.sitkFloat32)   # 例：動脈相    reg = sitk.ImageRegistrationMethod()    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)    reg.SetOptimizerAsRegularStepGradientDescent(1.0, 1e-4, 200)    reg.SetOptimizerScalesFromPhysicalShift()     # ★これを忘れると途中で止まる（後述）    reg.SetInitialTransform(sitk.CenteredTransformInitializer(        fixed, moving, sitk.Euler3DTransform(),        sitk.CenteredTransformInitializerFilter.GEOMETRY))    reg.SetInterpolator(sitk.sitkLinear)    tf = reg.Execute(fixed, moving)    tx, ty, tz = tf.GetParameters()[3:6]          # 平行移動の3成分[mm]    return (tx**2 + ty**2 + tz**2) ** 0.5         # ずれの大きさ[mm]

```text患者A ─┐患者B ─┼→ 同じアトラス空間へ → 全員が共通座標に載る患者C ─┘```

## そもそも、位置合わせが要らない比較もある

```text前回のCT  →  セグメンテーション  →  前回のマスク  →  体積 12 mL今回のCT  →  セグメンテーション  →  今回のマスク  →  体積 18 mL                                              → 6 mL 増えた（位置合わせ不要）```

## 中身は「似ている度合い」の最大化

In [ ]:
import SimpleITK as sitkfixed  = sitk.ReadImage("ct_今回.nii.gz", sitk.sitkFloat32)   # 基準にする画像（動かさない）moving = sitk.ReadImage("ct_前回.nii.gz", sitk.sitkFloat32)   # こちらを動かして合わせるreg = sitk.ImageRegistrationMethod()# CT同士だが、造影相の違いにも耐えるよう安全側で相互情報量を使うreg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)  # 似ている度合いの指標reg.SetOptimizerAsRegularStepGradientDescent(learningRate=1.0, minStep=1e-4,                                             numberOfIterations=200)reg.SetOptimizerScalesFromPhysicalShift()       # ★回転と平行移動の「効き」を揃える（下記）reg.SetShrinkFactorsPerLevel([4, 2, 1])         # 粗い解像度から順に合わせる（多重解像度）reg.SetSmoothingSigmasPerLevel([2, 1, 0])reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()reg.SetInitialTransform(sitk.CenteredTransformInitializer(    fixed, moving, sitk.Euler3DTransform(),                       # 剛体（移動＋回転の6自由度）    sitk.CenteredTransformInitializerFilter.GEOMETRY))reg.SetInterpolator(sitk.sitkLinear)transform = reg.Execute(fixed, moving)          # 変形パラメータを探すprint(transform.GetParameters())                 # 例: 回転3成分 + 平行移動3成分# 求めた変換で、前回の画像を今回の空間へ移すmoved = sitk.Resample(moving, fixed, transform, sitk.sitkLinear, 0.0)diff  = sitk.GetArrayFromImage(fixed) - sitk.GetArrayFromImage(moved)   # ← ここで初めて意味のある差分